# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

## GRUPO: NOMBRE DEL GRUPO

# Integrantes
* integrante
* integrante

### Ejercicio 1 - DLT ecuaciones formales

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 6.

### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



In [ ]:
import numpy as np;

def obtenerQ3d(imgPts, objPts):
    n = len(imgPts)
    Q = np.zeros((2*n, 12))

    for i in range(n):
        X, Y, Z = objPts[i]
        u, v = imgPts[i]

        Q[2*i, 0:4] = [-X, -Y, -Z, -1]
        Q[2*i, 8:12] = [u*X, u*Y, u*Z, u]

        Q[2*i+1, 4:8] = [-X, -Y, -Z, -1]
        Q[2*i+1, 8:12] = [v*X, v*Y, v*Z, v]

    return Q

objPts = np.loadtxt(' calib/3d/p_W_corners.txt', delimiter=',')
detected_all = np.loadtxt(' calib/3d/detected_corners.txt')  # Cargar matriz (120, 24)

# 2. Ahora sí extraemos el frame deseado y le hacemos reshape
frame_idx = 0
imgPts_frame = detected_all[frame_idx].reshape(-1, 2)

print(obtenerQ3d(imgPts, objPts));

import numpy as np

import numpy as np

def devolverMatricesCalibracion3d(M):
    M = np.asarray(M, dtype=float)

    # 1) Cortar M en A (3x3) y b (3x1)
    A = M[:, :3] # esto es KR
    b = M[:, 3] # Esto es Kt

    # 2) Corregir el signo: si det(A) < 0, M -> -M
    #    (det(A) = a1 . (a2 x a3), producto triple de las filas)
    if A[0] @ np.cross(A[1], A[2]) < 0:
        A = -A
        b = -b

    # 3) Filas de A
    a1, a2, a3 = A[0], A[1], A[2]

    K = np.zeros((3, 3))
    R = np.zeros((3, 3))

    # Fila 3
    K[2, 2] = np.linalg.norm(a3)
    R[2] = a3 / K[2, 2]

    # Fila 2
    K[1, 2] = a2 @ R[2]
    v = a2 - K[1, 2] * R[2]
    K[1, 1] = np.linalg.norm(v)
    R[1] = v / K[1, 1]

    # Fila 1
    K[0, 2] = a1 @ R[2]
    K[0, 1] = a1 @ R[1]
    v = a1 - K[0, 1] * R[1] - K[0, 2] * R[2]
    K[0, 0] = np.linalg.norm(v)
    R[0] = v / K[0, 0]

    # 4) Normalizar: K[2,2] = 1 (dividir K y b por el mismo valor)
    s = K[2, 2]
    K = K / s
    b = b / s

    # 5) Resolver K t = b por sustitución hacia atrás
    t = np.zeros(3)
    t[2] = b[2] / K[2, 2]
    t[1] = (b[1] - K[1, 2] * t[2]) / K[1, 1]
    t[0] = (b[0] - K[0, 1] * t[1] - K[0, 2] * t[2]) / K[0, 0]

    return R, t, K


ValueError: too many values to unpack (expected 2)

### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


a) Obtener la matriz $\mathbf{K}$

b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.

### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---